# Debugging CUDA Applications

This notebook covers common debugging techniques for CUDA-accelerated applications.

We start, as usual, by loading our ICE magic.

In [ ]:
%load_ext ice.magic

The first step is narrowing down potential problems as much as possible, in many cases down to a single kernel.
Having a systematic approach pays off here (check the [porting applications](./03-porting-applications.ipynb) notebook for additional details).
* A **CPU-only reference implementation** makes it possible to compare the output of individual functions, loop nests, or entire modules, which helps to pinpoint where the GPU version first diverges.
* A **parallel CPU implementation** - for instance using OpenMP - can further help isolate bugs that stem from parallelism rather than from the GPU offloading itself.

The techniques in this notebook assume that such an analysis has already been done and the problem has been isolated to a single kernel.

## Check for Missing Synchronization

Especially in complicated codes, it can be easy to forget important synchronization primitives such as `cudaDeviceSynchronize`.
An easy check is re-running the application with the environment variable `CUDA_LAUNCH_BLOCKING` set to `1`, i.e.

```bash
CUDA_LAUNCH_BLOCKING=1 ./my-app
```

This ensures that the CPU blocks after each kernel launch until it completes.
Note that the added synchronization can negatively impact performance.

## Check for Race Conditions

A good first diagnostic step is running the kernel with **serial execution**: a single thread across the entire grid.
If the serial result is correct but the parallel one is not, this strongly suggests a race condition.

To further narrow down the origin, vary the launch configuration systematically:
* Does the result stay correct with multiple blocks containing a single thread each?
* What about a single block with multiple threads?

This isolates whether the problem occurs between blocks or between threads within the same block (or both).

We have already encountered this pattern in [05-reductions.ipynb](./05-reductions.ipynb), where multiple threads concurrently modifying the same accumulator produced incorrect results.
A condensed version of that example is shown below.

In [ ]:
%%cuda -c code/debugging/faulty.cu

__global__ void reduce(int* acc, int numElements) {
    int start = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    
    for (int i = start; i < numElements; i += stride)
        acc[0] += 1;
           👆    👆
}

int main(int argc, char *argv[]) {
    int numElements = 64;

    int acc = 0;

    int *d_acc;
    checkCudaError(cudaMalloc(&d_acc, sizeof(int)));

    //# reset accumulator
    checkCudaError(cudaMemset(d_acc, 0, sizeof(int)));

    //# run reduction
    reduce<<<2, 32>>>(d_acc, numElements);

    checkCudaError(cudaDeviceSynchronize(), true);

    //# copy data back to host
    checkCudaError(cudaMemcpy(&acc, d_acc, sizeof(int), cudaMemcpyDeviceToHost));

    std::cout << "Accumulator: " << acc << " (should be " << numElements << ")" << std::endl;

    checkCudaError(cudaFree(d_acc));
}

## Print Debug Output

While generally considered a less elegant method, `printf` is a straightforward way to inspect runtime state from within a kernel.
Output is buffered and flushed after a kernel completes.
In case of multiple output sources (host, additional kernels), use `cudaDeviceSynchronize()` to make sure that you attribute output correctly.

Since threads execute in parallel, the output order is **non-deterministic**.
Including `blockIdx.x` and `threadIdx.x` in the format string makes it possible to identify which thread produced each line.
For an ordered output, launch the kernel with `<<<1, 1>>>`.

To keep the output manageable, it is usually advisable to reduce the number of threads and the problem size (or restrict output to anomalies).

Note that the default printf buffer is limited in size (1 MB).
For larger output, increase it with `cudaDeviceSetLimit(cudaLimitPrintfFifoSize, new_size)`.

In [ ]:
%%cuda -c code/debugging/faulty-printf.cu

__global__ void reduce(int* acc, int numElements) {
    int start = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    
    for (int i = start; i < numElements; i += stride) {
        printf("[block %d, thread %d] Processing element %d; Accumulator: %d\n", blockIdx.x, threadIdx.x, i, acc[0]);
        acc[0] += 1;
        printf("[block %d, thread %d] Updated Accumulator: %d\n", blockIdx.x, threadIdx.x, acc[0]);
    }
}

int main(int argc, char *argv[]) {
    int numElements = 64;

    int acc = 0;

    int *d_acc;
    checkCudaError(cudaMalloc(&d_acc, sizeof(int)));

    //# reset accumulator
    checkCudaError(cudaMemset(d_acc, 0, sizeof(int)));

    //# run reduction
    reduce<<<2, 32>>>(d_acc, numElements);

    checkCudaError(cudaDeviceSynchronize(), true);

    //# copy data back to host
    checkCudaError(cudaMemcpy(&acc, d_acc, sizeof(int), cudaMemcpyDeviceToHost));

    std::cout << "Accumulator: " << acc << " (should be " << numElements << ")" << std::endl;

    checkCudaError(cudaFree(d_acc));
}

## Assert in Kernels

`assert()` from `<cassert>` works inside CUDA kernels as well.
When an assertion fails, the thread halts and `cudaDeviceSynchronize()` returns an error.

Two useful options:
* Compile with `-G` (device debug info) for more detailed error messages.
  Note that this potentially decreases performance as it prevents certain optimizations.
* Assertions are automatically compiled out when `-DNDEBUG` is defined (typical for release builds).

The example below computes `sqrt` on an array.
One element is negative, which would produce `NaN` - the assertion catches this before it happens.

In [ ]:
%%cuda -c code/debugging/sqrt-assert.cu -v -f G

#include <cassert>

__global__ void sqrtKernel(float* data, int numElements) {
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < numElements; i += blockDim.x * gridDim.x) {
        assert(data[i] >= 0.);      //# triggers for negative values

        data[i] = sqrt(data[i]);
    }
}

int main() {
    const int numElements = 5;
    float h_data[] = {4., 9., -1., 16., 25.};       //# -1.0 triggers the assert

    float* d_data;
    checkCudaError(cudaMalloc(&d_data, numElements * sizeof(float)));
    checkCudaError(cudaMemcpy(d_data, h_data, numElements * sizeof(float), cudaMemcpyHostToDevice));

    sqrtKernel<<<1, 32>>>(d_data, numElements);
    checkCudaError(cudaDeviceSynchronize(), true);  //# reports assertion failure

    checkCudaError(cudaFree(d_data));
}

## Compute Sanitizer

`compute-sanitizer` is an external tool that runs a compiled CUDA application and detects errors that would otherwise go unnoticed at runtime.
The basic invocation is:

```bash
compute-sanitizer [options] app_name [app_options]
```

It provides several distinct checking mechanisms, each as a separate sub-tool:

* **Memcheck** (default) - detects erroneous memory accesses.
* **Initcheck** - detects reads of uninitialized global memory.
* **Racecheck** - detects **shared memory** data races.
* **Synccheck** - detects invalid uses of synchronization primitives within kernels.

The sub-tool is selected with `--tool <name>`, e.g. `compute-sanitizer --tool initcheck ./app`.
More information is available in the official [documentation](https://docs.nvidia.com/compute-sanitizer/ComputeSanitizer/index.html).

The kernel below writes each thread's index into an array, but launches 32 threads for an array of only 30 elements.
Without a bounds check, the last two threads write past the end of the allocated buffer.
At runtime, this goes undetected in most cases and the kernel completes without reported errors.

In [ ]:
%%cuda -c code/debugging/oob.cu

__global__ void fillArray(int* data, int numElements) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    //# BUG: if (idx < numElements) check is missing
    data[idx] = idx;
}

int main() {
    const int numElements = 30;

    int* d_data;
    checkCudaError(cudaMalloc(&d_data, numElements * sizeof(int)));

    fillArray<<<1, 32>>>(d_data, numElements);
    checkCudaError(cudaDeviceSynchronize(), true);

    std::cout << "Kernel completed" << std::endl;

    checkCudaError(cudaFree(d_data));
}

Running the application under `compute-sanitizer` (using the default `memcheck` tool) catches the out-of-bounds writes that the kernel produced silently.
The tool reports each invalid access together with the responsible thread.

In [ ]:
!compute-sanitizer $TMPDIR/app

The next example has no out-of-bounds accesses, so `memcheck` reports nothing.
Instead, it reads from uninitialized global memory - a class of bug that requires the `initcheck` tool to detect.

In [ ]:
%%cuda -c code/debugging/uninitialized.cu

__global__ void addOne(int* data, int numElements) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < numElements)
        data[idx] += 1;
}

int main() {
    const int numElements = 2;

    int* d_data;
    checkCudaError(cudaMalloc(&d_data, numElements * sizeof(int)));

    addOne<<<1, 32>>>(d_data, numElements);
    checkCudaError(cudaDeviceSynchronize(), true);

    std::cout << "Kernel completed" << std::endl;

    checkCudaError(cudaFree(d_data));
}

As expected, the default `memcheck` tool reports no errors.
Running the same application under `initcheck` detects the uninitialized reads: `cudaMalloc` does not zero-initialize, so the kernel reads indeterminate values.
The tool reports each uninitialized access together with the responsible thread.

In [ ]:
!compute-sanitizer $TMPDIR/app

In [ ]:
!compute-sanitizer --tool initcheck $TMPDIR/app

## CUDA GDB

`cuda-gdb` is a full-featured symbolic debugger for CUDA applications, built on top of GDB.
It supports all standard GDB commands and adds GPU-specific extensions.
The latter allows for switching between threads and blocks during a live debugging session.

To use it, compile with `-g` (host debug info) and `-G` (device debug info).
The `-G` flag disables optimizations on device code so that line-by-line stepping works correctly:

```bash
nvcc -g -G -o my-app my-code.cu
cuda-gdb ./my-app
```

A full walkthrough of `cuda-gdb` is beyond the scope of this course, but developers already familiar with GDB will feel largely at home.

## Next Step

Return to the [summary & outlook](./06-summary-outlook.ipynb) notebook.